In [85]:
import os
if os.path.basename(os.getcwd()) == "limpieza":
    os.chdir("..")
    
import pandas as pd
import numpy as np
from utils.funciones_filtrado import tipo_nulo_unicos_x_columna

In [86]:
base_ventas = pd.read_excel('ventas.xlsx')
base_ventas.head()

,venta_id,fecha,sucursal,producto,categoria,cantidad,precio_unitario,empleado,metodo_pago
0,V0228,2024-01-24,Puebla Norte,Brownie,Alimento,1,28,Ana P.,efectivo
1,V0064,10/08/2024,ANGELOPOLIS,Muffin Chocolate,Alimento,2,55,Sofia T.,Tarjeta
2,V0128,03/15/2024,puebla norte,Latte,Café,2,32,Diana F.,Tarjeta
3,V0137,09/17/2024,Puebla Centro,Espresso,Café,4,65,Ana P.,TARJETA
4,V0023,08/20/2024,San Andrés,Muffin Chocolate,Alimento,1,18,Carlos R.,transferencia


Primero vamos a eliminar las ordenes repetidas, ya que así optimizamos un poco más los siguientes flujos al tener menos filas.
Ya que hay registros duplicados en las que unas filas tienen valores nulos y otros no, lo que se hará es tomar aquellas filas que tengan valores y así evitar traer valores nulos.

In [87]:
base_ventas = base_ventas.groupby('venta_id', as_index = False).first()
base_ventas

,venta_id,fecha,sucursal,producto,categoria,cantidad,precio_unitario,empleado,metodo_pago
0,V0001,2024-11-23,Puebla Centro,Muffin Chocolate,Alimento,5,65,María G.,Efectivo
1,V0002,2024-01-17,Pue. Centro,Latte Vainilla,Café,5,55,Pedro H.,TARJETA
2,V0003,08/17/2024,Puebla Centro,Croissant,Alimento,3,145,María G.,Tarjeta
3,V0004,2024-07-13,ANGELOPOLIS,Pay de Queso,Alimento,5,35,María G.,Efectivo
4,V0005,02/10/2024,Cho.,Té Negro,Té,2,30,Luis M.,Tarjeta
...,...,...,...,...,...,...,...,...,...
395,V0396,06/01/2024,Puebla Centro,Té Verde,Té,5,$28,Pedro H.,Tarjeta
396,V0397,05/06/2024,ANGELOPOLIS,Latte Vainilla,Café,3,40,Ana P.,Tarjeta
397,V0398,2024-04-02,Cholula,Frappé Caramelo,Frío,1,28,Ana P.,Tarjeta
398,V0399,06/18/2024,Puebla Centro,Muffin Arándano,Alimento,2,55,Sofia T.,Tarjeta


Ahora vamos a ver como quedo nuestra tabla despues de la limpieza

In [88]:
print(f'La longitud de la tabla es: {len(base_ventas)}')
inspeccion = tipo_nulo_unicos_x_columna(base_ventas)
inspeccion

La longitud de la tabla es: 400


,columna,tipo de dato,cantidad de valores nulos,cantidad de valores unicos
0,venta_id,object,0,400
1,fecha,object,0,332
2,sucursal,object,0,21
3,producto,object,0,15
4,categoria,object,0,4
5,cantidad,int64,0,5
6,precio_unitario,object,0,46
7,empleado,object,0,8
8,metodo_pago,object,0,6


De esta manera notamos que ya no hay ningún valor nulo, ni id's de ventas repetidos.
Por lo que ahora vamos a limpiar y a estandarizar las sucursales.

In [89]:
base_ventas['sucursal'] = base_ventas['sucursal'].str.lower().str.strip() 

base_ventas['sucursal'] = np.where(base_ventas['sucursal'].str.startswith('ang'), 'angelopolis', base_ventas['sucursal'])
base_ventas['sucursal'] = np.where(base_ventas['sucursal'].str.endswith('norte'), 'puebla norte', base_ventas['sucursal'])
base_ventas['sucursal'] = np.where(base_ventas['sucursal'].str.endswith('centro'), 'puebla centro', base_ventas['sucursal'])
base_ventas['sucursal'] = np.where(base_ventas['sucursal'].str.startswith('s'), 'san andres', base_ventas['sucursal'])
base_ventas['sucursal'] = np.where(base_ventas['sucursal'].str.startswith('c'), 'cholula', base_ventas['sucursal'])

base_ventas['metodo_pago'] = base_ventas['metodo_pago'].str.lower().str.strip() #también aprovechamos a estandirizar esta columna

base_ventas['sucursal'].unique() #con esto corroborramos que ya sólo tenemos las 5 sucursales

array(['puebla centro', 'angelopolis', 'cholula', 'puebla norte',
       'san andres'], dtype=object)

In [90]:
base_ventas['precio_unitario'] = base_ventas['precio_unitario'].astype(str) #aseguramos el tipo que sea cadena
base_ventas['precio_unitario'] = base_ventas['precio_unitario'].str.replace('$', '', regex = False) #quitamos '$'
base_ventas['precio_unitario'] = base_ventas['precio_unitario'].str.replace('USD', '', regex = False) #quitamos 'USD'

base_ventas['precio_unitario'] = pd.to_numeric(base_ventas['precio_unitario'], errors='coerce')

Empezamos con la limpieza de las fechas

In [91]:
base_ventas['fecha'] = base_ventas['fecha'].astype(str).str.replace(' de ', ' ', regex=False) 
#reemplazamos el patrón ´ de ' con un espacio
#esto con el motivo de tener un estandar, ya que python puede interpretar diferentes tipos de fechas
#por lo que el primer patron sería aaaa-mm-dd, mm/dd/aaaa y el último de dd Month aaaa
base_ventas['fecha'] = pd.to_datetime(base_ventas['fecha'], format='mixed', errors='coerce')

In [92]:
print(f'La longitud de la tabla es: {len(base_ventas)}')
inspeccion = tipo_nulo_unicos_x_columna(base_ventas)
inspeccion

La longitud de la tabla es: 400


,columna,tipo de dato,cantidad de valores nulos,cantidad de valores unicos
0,venta_id,object,0,400
1,fecha,datetime64[ns],0,239
2,sucursal,object,0,5
3,producto,object,0,15
4,categoria,object,0,4
5,cantidad,int64,0,5
6,precio_unitario,int64,0,30
7,empleado,object,0,8
8,metodo_pago,object,0,3


In [93]:
base_ventas.to_excel('ventas_limpio.xlsx', index=False)